# Preprocesamiento: Breast Cancer Wisconsin (Diagnostic) Dataset

## 🎯 Objetivos de Aprendizaje
- **Técnica Principal**: Clasificación Binaria + Validación Cruzada Robusta
- **Dataset**: Breast Cancer Wisconsin (30 características)
- **Nivel**: Intermedio (transición entre Wine Quality y Forest Cover Type)

## 📊 Contexto del Dataset
Este dataset contiene mediciones de características de núcleos celulares tomadas de imágenes digitales de aspirados con aguja fina (FNA) de masas mamarias. Es ideal para:
- **Clasificación binaria** (Benigno vs Maligno)
- **Validación cruzada** con métricas médicas
- **Preprocesamiento de características continuas**
- **Comparación de algoritmos** de clasificación

## 💡 ¿Por qué Breast Cancer como "Bono"?
- ✅ Dataset limpio (0% valores faltantes)
- ✅ Todas las características son numéricas continuas
- ✅ Familiar en el dominio médico
- ❌ No introduce técnicas NUEVAS significativas vs Wine Quality
- ❌ Ideal como práctica adicional, no como "próximo paso" principal


In [ ]:
# 📦 Importaciones
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
from sklearn.metrics import precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8')
sns.set_palette("Set2")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

## 🔄 Paso 1: Carga y Exploración Inicial

**¿Por qué empezar simple?** Este dataset es perfecto para validar que entendemos los conceptos básicos antes de pasar a datasets más complejos.

In [ ]:
# Cargar el dataset
print("🧬 Cargando Breast Cancer Wisconsin Dataset...")
cancer_data = load_breast_cancer()

# Convertir a DataFrame para análisis
X = pd.DataFrame(cancer_data.data, columns=cancer_data.feature_names)
y = pd.Series(cancer_data.target, name='diagnosis')

# Mapear valores del target a nombres más claros
y_mapped = y.map({0: 'Malignant', 1: 'Benign'})

# Información básica
print(f"📊 Shape del dataset: {X.shape}")
print(f"🎯 Número de características: {X.shape[1]}")
print(f"🎯 Número de clases: {len(cancer_data.target_names)}")
print(f"📈 Distribución de clases:")
class_counts = pd.Series(cancer_data.target).value_counts().sort_index()
for i, class_name in enumerate(cancer_data.target_names):
    count = class_counts[i]
    percentage = count / len(y) * 100
    print(f"  - {class_name}: {count} casos ({percentage:.1f}%)")

# Verificar valores faltantes
missing_values = X.isnull().sum()
print(f"\n🔍 Valores faltantes: {missing_values.sum()} total")
if missing_values.sum() == 0:
    print("✅ Dataset completamente limpio - sin valores faltantes")
else:
    print("⚠️ Encontrados valores faltantes:")
    print(missing_values[missing_values > 0])

# Información de tipos de datos
print(f"\n📊 Tipos de datos:")
print(X.dtypes.value_counts())
print(f"\n🏷️ Todas las características son numéricas continuas ✅")

# Mostrar las primeras características
print(f"\n📋 Primeras 10 características:")
for i, feature in enumerate(X.columns[:10]):
    print(f"  {i+1:2d}. {feature}")

## 📊 Paso 2: Análisis Exploratorio de Datos (EDA)

**🎯 Enfoque**: Análisis de distribución y separación de clases.

In [ ]:
# Análisis de distribuciones por clase
print("📊 Análisis de distribuciones por clase")

# Estadísticas descriptivas
print(f"\n📈 Estadísticas descriptivas:")
desc_stats = X.describe()
print(desc_stats.round(3))

# Análisis de algunas características clave
key_features = [
    'mean radius', 'mean texture', 'mean perimeter', 'mean area',
    'mean smoothness', 'mean compactness', 'mean concavity',
    'worst radius', 'worst area', 'worst concavity'
]

# Comparar estadísticas por clase
print(f"\n🔍 Comparación de características clave por clase:")
comparison_stats = pd.DataFrame()

for feature in key_features[:5]:  # Primeras 5 para no saturar
    if feature in X.columns:
        benign_stats = X[y == 1][feature].describe()
        malignant_stats = X[y == 0][feature].describe()
        
        comparison_stats[feature] = [
            benign_stats['mean'], benign_stats['std'],
            malignant_stats['mean'], malignant_stats['std']
        ]

comparison_stats.index = ['Benign_Mean', 'Benign_Std', 'Malignant_Mean', 'Malignant_Std']
print(comparison_stats.round(3))

# Visualización de distribuciones
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

key_features_viz = ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness', 'worst radius']

for i, feature in enumerate(key_features_viz):
    if feature in X.columns:
        # Histograma por clase
        axes[i].hist(X[y == 1][feature], alpha=0.7, label='Benign', bins=30, density=True)
        axes[i].hist(X[y == 0][feature], alpha=0.7, label='Malignant', bins=30, density=True)
        axes[i].set_title(f'{feature}')
        axes[i].set_xlabel(feature)
        axes[i].set_ylabel('Density')
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle('Distribución de Características por Tipo de Tumor', y=1.02)
plt.show()

# Análisis de separabilidad
print(f"\n🎯 Análisis de separabilidad:")
for feature in ['mean radius', 'mean area', 'worst radius']:
    if feature in X.columns:
        benign_values = X[y == 1][feature]
        malignant_values = X[y == 0][feature]
        
        # Calcular solapamiento
        overlap = len(set(benign_values) & set(malignant_values))
        total_unique = len(set(benign_values) | set(malignant_values))
        
        print(f"  {feature}:")
        print(f"    - Benign: μ={benign_values.mean():.2f}, σ={benign_values.std():.2f}")
        print(f"    - Malignant: μ={malignant_values.mean():.2f}, σ={malignant_values.std():.2f}")
        print(f"    - Diferencia de medias: {abs(benign_values.mean() - malignant_values.mean()):.2f}")
        print(f"    - Solapamiento: {overlap/total_unique*100:.1f}%")
        print()

## 🔗 Paso 3: Análisis de Correlaciones

**🎯 Objetivo**: Identificar características altamente correlacionadas que puedan afectar el rendimiento del modelo.

In [ ]:
# Matriz de correlación
print("🔗 Análisis de correlaciones")

# Calcular matriz de correlación
correlation_matrix = X.corr()

# Encontrar correlaciones muy altas (>0.9)
high_corr_threshold = 0.9
high_corr_pairs = []

for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        corr_value = abs(correlation_matrix.iloc[i, j])
        if corr_value > high_corr_threshold:
            col1 = correlation_matrix.columns[i]
            col2 = correlation_matrix.columns[j]
            high_corr_pairs.append((col1, col2, correlation_matrix.iloc[i, j]))

print(f"\n🔍 Encontradas {len(high_corr_pairs)} pares con correlación > {high_corr_threshold}:")
for col1, col2, corr in sorted(high_corr_pairs, key=lambda x: abs(x[2]), reverse=True)[:15]:
    print(f"  {col1} ↔ {col2}: {corr:.3f}")

# Visualización de matriz de correlación
plt.figure(figsize=(16, 14))

# Crear máscara para la diagonal superior
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

# Heatmap de correlaciones
sns.heatmap(correlation_matrix, mask=mask, annot=False, cmap='RdBu_r', 
            center=0, square=True, linewidths=0.5, cbar_kws={"shrink": .8})
plt.title('Matriz de Correlación - Breast Cancer Features')
plt.tight_layout()
plt.show()

# Identificar grupos de características altamente correlacionadas
print(f"\n📊 Grupos de características correlacionadas:")
feature_groups = {
    'radius_group': [col for col in X.columns if 'radius' in col],
    'texture_group': [col for col in X.columns if 'texture' in col],
    'perimeter_group': [col for col in X.columns if 'perimeter' in col],
    'area_group': [col for col in X.columns if 'area' in col],
    'smoothness_group': [col for col in X.columns if 'smoothness' in col],
    'compactness_group': [col for col in X.columns if 'compactness' in col],
    'concavity_group': [col for col in X.columns if 'concavity' in col],
    'symmetry_group': [col for col in X.columns if 'symmetry' in col],
    'fractal_group': [col for col in X.columns if 'fractal' in col]
}

for group_name, features in feature_groups.items():
    if len(features) > 1:
        print(f"  {group_name}: {len(features)} características")
        print(f"    {features[:3]}{'...' if len(features) > 3 else ''}")

print(f"\n💡 Nota: Alta correlación dentro de grupos es esperada")
print(f"   (ej: 'mean radius', 'worst radius', 'radius error')")

## 🔄 Paso 4: Preparación de Datos

**🎯 Enfoque**: Preprocesamiento estándar antes de aplicar modelos.

In [ ]:
# Preparación de datos
print("🔄 Preparación de datos")

# División en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"📊 División de datos:")
print(f"  - Entrenamiento: {X_train.shape[0]} muestras ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"  - Prueba: {X_test.shape[0]} muestras ({X_test.shape[0]/len(X)*100:.1f}%)")

# Verificar distribución de clases en splits
print(f"\n📈 Distribución de clases en entrenamiento:")
train_dist = y_train.value_counts()
test_dist = y_test.value_counts()

for class_idx, class_name in enumerate(['Malignant', 'Benign']):
    train_pct = train_dist[class_idx] / len(y_train) * 100
    test_pct = test_dist[class_idx] / len(y_test) * 100
    print(f"  - {class_name}: Train={train_pct:.1f}%, Test={test_pct:.1f}%")

# Escalado de características
print(f"\n⚖️ Escalado de características")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convertir de vuelta a DataFrame para mantener nombres de columnas
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

# Verificar el efecto del escalado
print(f"\n📊 Efecto del escalado (primeras 5 características):")
before_scaling = X_train.iloc[:, :5].describe().round(3)
after_scaling = X_train_scaled.iloc[:, :5].describe().round(3)

print("ANTES del escalado:")
print(before_scaling)
print("\nDESPUÉS del escalado:")
print(after_scaling)

print(f"\n✅ Datos preparados para modelado")
print(f"   - Shape entrenamiento: {X_train_scaled.shape}")
print(f"   - Shape prueba: {X_test_scaled.shape}")
print(f"   - Todas las características tienen media ≈ 0 y std ≈ 1")

## 🤖 Paso 5: Comparación de Modelos con Validación Cruzada

**🎯 Objetivo**: Comparar diferentes algoritmos de clasificación para encontrar el mejor rendimiento.

In [ ]:
# Definir modelos a comparar
print("🤖 Comparación de Modelos de Clasificación")

models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5)
}

print(f"\n📋 Modelos a evaluar: {list(models.keys())}")

# Validación cruzada estratificada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

print(f"\n🔄 Ejecutando validación cruzada (5-fold estratificado)...")

for model_name, model in models.items():
    print(f"  Evaluando {model_name}...")
    
    # Accuracy
    cv_accuracy = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
    
    # ROC AUC
    cv_roc_auc = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='roc_auc')
    
    # Precision (clase positiva = 1 = Benign)
    cv_precision = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='precision')
    
    # Recall (clase positiva = 1 = Benign)
    cv_recall = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='recall')
    
    # F1 Score
    cv_f1 = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='f1')
    
    results[model_name] = {
        'accuracy': cv_accuracy,
        'roc_auc': cv_roc_auc,
        'precision': cv_precision,
        'recall': cv_recall,
        'f1': cv_f1
    }

# Mostrar resultados
print(f"\n📊 Resultados de Validación Cruzada (Media ± Desviación Estándar):")
print("=" * 85)
print(f"{'Modelo':<20} {'Accuracy':<12} {'ROC AUC':<12} {'Precision':<12} {'Recall':<12} {'F1':<12}")
print("-" * 85)

for model_name, metrics in results.items():
    acc_mean = metrics['accuracy'].mean()
    acc_std = metrics['accuracy'].std()
    auc_mean = metrics['roc_auc'].mean()
    auc_std = metrics['roc_auc'].std()
    prec_mean = metrics['precision'].mean()
    prec_std = metrics['precision'].std()
    rec_mean = metrics['recall'].mean()
    rec_std = metrics['recall'].std()
    f1_mean = metrics['f1'].mean()
    f1_std = metrics['f1'].std()
    
    print(f"{model_name:<20} {acc_mean:.3f}±{acc_std:.3f}  {auc_mean:.3f}±{auc_std:.3f}  "
          f"{prec_mean:.3f}±{prec_std:.3f}  {rec_mean:.3f}±{rec_std:.3f}  {f1_mean:.3f}±{f1_std:.3f}")

# Identificar mejor modelo
best_model_name = max(results.keys(), key=lambda x: results[x]['accuracy'].mean())
best_accuracy = results[best_model_name]['accuracy'].mean()

print(f"\n🏆 Mejor modelo: {best_model_name}")
print(f"   Accuracy promedio: {best_accuracy:.3f} ± {results[best_model_name]['accuracy'].std():.3f}")
print(f"   ROC AUC promedio: {results[best_model_name]['roc_auc'].mean():.3f} ± {results[best_model_name]['roc_auc'].std():.3f}")

In [ ]:
# Visualización de resultados
print("📈 Visualización de resultados de validación cruzada")

# Crear DataFrame para visualización
viz_data = []
for model_name, metrics in results.items():
    for fold_idx in range(5):
        viz_data.append({
            'Modelo': model_name,
            'Fold': fold_idx + 1,
            'Accuracy': metrics['accuracy'][fold_idx],
            'ROC_AUC': metrics['roc_auc'][fold_idx],
            'Precision': metrics['precision'][fold_idx],
            'F1_Score': metrics['f1'][fold_idx]
        })

viz_df = pd.DataFrame(viz_data)

# Gráficos de comparación
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Boxplot de accuracy
sns.boxplot(data=viz_df, x='Modelo', y='Accuracy', ax=axes[0,0])
axes[0,0].set_title('Distribución de Accuracy por Modelo')
axes[0,0].tick_params(axis='x', rotation=45)
axes[0,0].grid(True, alpha=0.3)

# Boxplot de ROC AUC
sns.boxplot(data=viz_df, x='Modelo', y='ROC_AUC', ax=axes[0,1])
axes[0,1].set_title('Distribución de ROC AUC por Modelo')
axes[0,1].tick_params(axis='x', rotation=45)
axes[0,1].grid(True, alpha=0.3)

# Lineplot de accuracy por fold
for model in viz_df['Modelo'].unique():
    model_data = viz_df[viz_df['Modelo'] == model]
    axes[1,0].plot(model_data['Fold'], model_data['Accuracy'], 'o-', label=model, linewidth=2, markersize=6)
axes[1,0].set_xlabel('Fold')
axes[1,0].set_ylabel('Accuracy')
axes[1,0].set_title('Accuracy por Fold de Validación Cruzada')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)
axes[1,0].set_xticks(range(1, 6))

# Comparación de métricas promedio
metrics_avg = []
for model_name in results.keys():
    metrics_avg.append({
        'Modelo': model_name,
        'Accuracy': results[model_name]['accuracy'].mean(),
        'ROC_AUC': results[model_name]['roc_auc'].mean(),
        'Precision': results[model_name]['precision'].mean(),
        'Recall': results[model_name]['recall'].mean(),
        'F1': results[model_name]['f1'].mean()
    })

metrics_df = pd.DataFrame(metrics_avg)
metrics_melted = metrics_df.melt(id_vars=['Modelo'], 
                                 value_vars=['Accuracy', 'ROC_AUC', 'Precision', 'Recall', 'F1'],
                                 var_name='Métrica', 
                                 value_name='Valor')

sns.barplot(data=metrics_melted, x='Métrica', y='Valor', hue='Modelo', ax=axes[1,1])
axes[1,1].set_title('Comparación de Métricas Promedio')
axes[1,1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
axes[1,1].grid(True, alpha=0.3)
plt.setp(axes[1,1].get_xticklabels(), rotation=45)

plt.tight_layout()
plt.show()

## 🔍 Paso 6: Evaluación Detallada del Mejor Modelo

**🎯 Enfoque**: Análisis profundo del rendimiento del mejor modelo en el conjunto de prueba.

In [ ]:
# Evaluación detallada del mejor modelo
print(f"🔍 Evaluación Detallada del Mejor Modelo: {best_model_name}")

# Entrenar el mejor modelo con todo el conjunto de entrenamiento
best_model = models[best_model_name]
best_model.fit(X_train_scaled, y_train)

# Predicciones
y_pred = best_model.predict(X_test_scaled)
y_pred_proba = best_model.predict_proba(X_test_scaled)[:, 1]  # Probabilidad de clase positiva (Benign)

# Métricas de evaluación
test_accuracy = accuracy_score(y_test, y_pred)
test_roc_auc = roc_auc_score(y_test, y_pred_proba)
test_precision = precision_score(y_test, y_pred)
test_recall = recall_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred)

print(f"\n📊 Métricas en Conjunto de Prueba:")
print(f"{'Métrica':<15} {'Valor':<10} {'Interpretación'}")
print("-" * 50)
print(f"{'Accuracy':<15} {test_accuracy:.3f}      Proporción de predicciones correctas")
print(f"{'ROC AUC':<15} {test_roc_auc:.3f}      Capacidad de discriminación")
print(f"{'Precision':<15} {test_precision:.3f}      Precisión (VP / VP + FP)")
print(f"{'Recall':<15} {test_recall:.3f}      Sensibilidad (VP / VP + FN)")
print(f"{'F1 Score':<15} {test_f1:.3f}      Media armónica de precision y recall")

# Clasificación detallada
print(f"\n📋 Reporte de Clasificación Completo:")
target_names = ['Malignant', 'Benign']
print(classification_report(y_test, y_pred, target_names=target_names))

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(12, 5))

# Subplot 1: Matriz de confusión
plt.subplot(1, 2, 1)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=target_names, yticklabels=target_names)
plt.title(f'Matriz de Confusión - {best_model_name}')
plt.xlabel('Predicción')
plt.ylabel('Valor Real')

# Interpretación de errores
tn, fp, fn, tp = cm.ravel()
print(f"\n🔍 Análisis de Errores:")
print(f"  - Verdaderos Negativos (TN): {tn} = Malignant correctamente clasificado")
print(f"  - Falsos Positivos (FP): {fp} = Malignant mal clasificado como Benign")
print(f"  - Falsos Negativos (FN): {fn} = Benign mal clasificado como Malignant")
print(f"  - Verdaderos Positivos (TP): {tp} = Benign correctamente clasificado")

print(f"\n⚠️ Implicaciones Médicas:")
print(f"  - Falsos Positivos ({fp}): Pacientes sanos diagnosticados como enfermos")
print(f"    → Puede llevar a pruebas adicionales innecesarias")
print(f"  - Falsos Negativos ({fn}): Pacientes enfermos no detectados")
print(f"    → Más grave médicamente (diagnóstico perdido)")

# Subplot 2: Distribución de probabilidades
plt.subplot(1, 2, 2)
plt.hist(y_pred_proba[y_test == 0], alpha=0.7, label='Malignant', bins=20, density=True)
plt.hist(y_pred_proba[y_test == 1], alpha=0.7, label='Benign', bins=20, density=True)
plt.axvline(x=0.5, color='red', linestyle='--', label='Umbral (0.5)')
plt.xlabel('Probabilidad de Clase Benign')
plt.ylabel('Densidad')
plt.title('Distribución de Probabilidades Predichas')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Análisis de casos mal clasificados
print(f"\n🔍 Casos Mal Clasificados:")
misclassified_mask = y_test != y_pred
misclassified_indices = np.where(misclassified_mask)[0]
test_indices = X_test.index

print(f"Total de casos mal clasificados: {len(misclassified_indices)}")

if len(misclassified_indices) > 0:
    print(f"\nDetalles de casos mal clasificados:")
    for i, idx in enumerate(misclassified_indices[:5]):  # Mostrar primeros 5
        test_idx = test_indices[idx]
        actual_class = target_names[int(y_test.iloc[idx])]
        predicted_class = target_names[int(y_pred[idx])]
        prob = y_pred_proba[idx]
        
        print(f"  Caso {i+1}: Índice {test_idx}")
        print(f"    Clase real: {actual_class}, Predicha: {predicted_class}")
        print(f"    Probabilidad: {prob:.3f}")
        
        # Características más importantes para este caso
        if hasattr(best_model, 'feature_importances_'):
            importances = best_model.feature_importances_
            top_features_idx = np.argsort(importances)[-3:][::-1]
            top_features = [X_train.columns[j] for j in top_features_idx]
            top_values = [X_test_scaled.iloc[idx, j] for j in top_features_idx]
            print(f"    Top 3 características: {', '.join([f'{f}({v:.2f})' for f, v in zip(top_features, top_values)])}")
        print()

## 📊 Paso 7: Interpretación del Modelo

**🎯 Objetivo**: Entender qué características son más importantes para la clasificación.

In [ ]:
# Interpretación del modelo (si es posible)
print(f"📊 Interpretación del Modelo: {best_model_name}")

if hasattr(best_model, 'feature_importances_'):
    # Random Forest: importancias de características
    print(f"\n🎯 Importancias de Características (Top 15):")
    
    feature_importance = pd.DataFrame({
        'feature': X_train.columns,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print(feature_importance.head(15).to_string(index=False))
    
    # Visualización de importancias
    plt.figure(figsize=(12, 8))
    top_15_features = feature_importance.head(15)
    
    bars = plt.barh(range(len(top_15_features)), top_15_features['importance'])
    plt.yticks(range(len(top_15_features)), top_15_features['feature'])
    plt.xlabel('Importancia')
    plt.title(f'Top 15 Características Más Importantes - {best_model_name}')
    plt.gca().invert_yaxis()
    
    # Agregar valores en las barras
    for i, bar in enumerate(bars):
        width = bar.get_width()
        plt.text(width + 0.001, bar.get_y() + bar.get_height()/2, 
                f'{width:.3f}', ha='left', va='center')
    
    plt.tight_layout()
    plt.show()
    
    # Análisis de características por grupo
    print(f"\n📊 Análisis de Importancia por Grupo de Características:")
    
    groups = {
        'mean': [f for f in X_train.columns if 'mean' in f],
        'worst': [f for f in X_train.columns if 'worst' in f],
        'error': [f for f in X_train.columns if 'error' in f]
    }
    
    group_importance = {}
    for group_name, features in groups.items():
        group_features = feature_importance[feature_importance['feature'].isin(features)]
        group_importance[group_name] = group_features['importance'].sum()
    
    print(f"{'Grupo':<10} {'Total Importancia':<20} {'Características':<30}")
    print("-" * 65)
    for group_name, importance in sorted(group_importance.items(), key=lambda x: x[1], reverse=True):
        n_features = len(groups[group_name])
        print(f"{group_name:<10} {importance:.3f}            {n_features} características")

elif hasattr(best_model, 'coef_'):
    # Logistic Regression: coeficientes
    print(f"\n🎯 Coeficientes del Modelo (Top 15 por magnitud):")
    
    coef_importance = pd.DataFrame({
        'feature': X_train.columns,
        'coefficient': np.abs(best_model.coef_[0])  # Valor absoluto
    }).sort_values('coefficient', ascending=False)
    
    print(coef_importance.head(15).to_string(index=False))
    
    # Visualización de coeficientes
    plt.figure(figsize=(12, 8))
    top_15_coef = coef_importance.head(15)
    
    bars = plt.barh(range(len(top_15_coef)), top_15_coef['coefficient'])
    plt.yticks(range(len(top_15_coef)), top_15_coef['feature'])
    plt.xlabel('Coeficiente (Valor Absoluto)')
    plt.title(f'Top 15 Características con Mayor Influencia - {best_model_name}')
    plt.gca().invert_yaxis()
    
    plt.tight_layout()
    plt.show()

else:
    print(f"\n⚠️ El modelo {best_model_name} no proporciona interpretación directa de características")
    print(f"   Considera usar SHAP o LIME para interpretación post-hoc")

# Resumen de insights médicos
print(f"\n💡 Insights Médicos Interpretados:")
if hasattr(best_model, 'feature_importances_'):
    top_features = feature_importance.head(5)['feature'].tolist()
    print(f"\nLas 5 características más importantes son:")
    for i, feature in enumerate(top_features, 1):
        if 'worst' in feature:
            feature_type = "peor caso"
        elif 'mean' in feature:
            feature_type = "valor promedio"
        elif 'error' in feature:
            feature_type = "error/variación"
        else:
            feature_type = "característica"
        
        print(f"  {i}. {feature.replace('_', ' ')} ({feature_type})")
        
        # Interpretación específica
        if 'radius' in feature:
            print(f"     → Radio promedio del núcleo celular")
        elif 'area' in feature:
            print(f"     → Área promedio del núcleo celular")
        elif 'perimeter' in feature:
            print(f"     → Perímetro promedio del núcleo celular")
        elif 'texture' in feature:
            print(f"     → Textura de la superficie celular")
        elif 'smoothness' in feature:
            print(f"     → Suavidad/local smoothness del contorno")
        elif 'compactness' in feature:
            print(f"     → Compacidad del núcleo (perímetro²/área - 1)")
        elif 'concavity' in feature:
            print(f"     → Severidad de porciones cóncavas del contorno")
        elif 'symmetry' in feature:
            print(f"     → Simetría del núcleo celular")
        elif 'fractal' in feature:
            print(f"     → Dimensión fractal del contorno")

print(f"\n🏥 Interpretación Clínica:")
print(f"   - Las características más importantes coinciden con observación médica:")
print(f"     • Núcleos celulares más grandes (radius/area) → más probables de ser malignos")
print(f"     • Formas más irregulares (concavity/compactness) → malignas")
print(f"     • Texturas más variables → malignas")
print(f"   - 'Worst' features (peores casos) tienden a ser más discriminativas")

## 🎓 Paso 8: Conclusiones y Recomendaciones

**🎯 Síntesis**: Evaluación del aprendizaje y siguientes pasos.

In [ ]:
# Conclusiones finales
print("🎓 CONCLUSIONES DEL ANÁLISIS DE BREAST CANCER DATASET")
print("=" * 60)

print(f"\n📊 RESUMEN DEL DATASET:")
print(f"  - Características: {X.shape[1]} (todas numéricas continuas)")
print(f"  - Muestras: {X.shape[0]}")
print(f"  - Clases: Binaria (Malignant: {len(y[y==0])}, Benign: {len(y[y==1])})")
print(f"  - Valores faltantes: {missing_values.sum()} (0%)")
print(f"  - Complejidad: Baja-Media")

print(f"\n🤖 RESULTADOS DEL MEJOR MODELO:")
print(f"  - Algoritmo: {best_model_name}")
print(f"  - Accuracy (CV): {results[best_model_name]['accuracy'].mean():.3f} ± {results[best_model_name]['accuracy'].std():.3f}")
print(f"  - Accuracy (Test): {test_accuracy:.3f}")
print(f"  - ROC AUC: {test_roc_auc:.3f}")
print(f"  - Precision: {test_precision:.3f}")
print(f"  - Recall: {test_recall:.3f}")
print(f"  - F1 Score: {test_f1:.3f}")

print(f"\n📈 COMPARACIÓN DE MODELOS:")
model_ranking = sorted(results.items(), key=lambda x: x[1]['accuracy'].mean(), reverse=True)
for i, (model_name, metrics) in enumerate(model_ranking, 1):
    print(f"  {i}. {model_name}: {metrics['accuracy'].mean():.3f} ± {metrics['accuracy'].std():.3f}")

print(f"\n💡 APRENDIZAJES CLAVE:")
print(f"\n✅ Fortalezas de este dataset:")
print(f"  1. Dataset muy limpio (ideal para aprender conceptos básicos)")
print(f"  2. Todas las características numéricas (no requiere encoding)")
print(f"  3. Clasificación binaria (más simple que multiclase)")
print(f"  4. Relevancia médica (contexto interesante)")
print(f"  5. Resultados interpretables (coeficientes/importancias)")

print(f"\n⚠️ Limitaciones para el aprendizaje:")
print(f"  1. No introduce técnicas nuevas vs Wine Quality")
print(f"  2. Muy pocas características categóricas para practicar")
print(f"  3. No hay valores faltantes para practicar imputación")
print(f"  4. Baja dimensionalidad (solo 30 features)")
print(f"  5. No hay outliers significativos")

print(f"\n🎯 RECOMENDACIONES DE USO EN CURSO:")
print(f"\n✅ Óptimo como:")
print(f"  - Práctica adicional después de Wine Quality")
print(f"  - Transición suave hacia técnicas más avanzadas")
print(f"  - Validación de conceptos de validación cruzada")
print(f"  - Comparación de algoritmos de clasificación")

print(f"\n❌ No recomendado como 'próximo paso principal' porque:")
print(f"  - Repite técnicas ya aprendidas en Wine Quality")
print(f"  - No prepara para desafíos como alta dimensionalidad")
print(f"  - Falta práctica con strings y valores faltantes")

print(f"\n🚀 SIGUIENTES PASOS SUGERIDOS:")
print(f"\n1. Inmediato:")
print(f"   • Probar hiperparámetros del mejor modelo con GridSearch")
print(f"   • Implementar ensemble de los mejores modelos")
print(f"   • Analizar sensibilidad a diferentes umbrales de clasificación")

print(f"\n2. Próximo dataset sugerido:")
print(f"   • Forest Cover Type (alta dimensionalidad + feature selection)")
print(f"   • Adult Census (strings + valores faltantes)")
print(f"   • Heart Disease (imputación avanzada)")

print(f"\n3. Técnicas avanzadas a explorar:")
print(f"   • Feature selection en datasets de alta dimensionalidad")
print(f"   • Manejo de datos categóricos y textuales")
print(f"   • Técnicas de imputación avanzada")
print(f"   • Análisis de componentes principales (PCA)")
print(f"   • Modelos de ensemble más sofisticados")

print(f"\n📚 PROGRESIÓN SUGERIDA DEL CURSO:")
print(f"  1. Wine Quality (13 features) → ⭐ Básicos")
print(f"  2. Breast Cancer (30 features) → ⭐ Práctica adicional")
print(f"  3. Forest Cover (54 features) → ⭐⭐⭐ Alta dimensionalidad")
print(f"  4. Adult Census → ⭐⭐ Strings + Missing values")
print(f"  5. Heart Disease → ⭐⭐⭐ Imputación avanzada")

print(f"\n✅ CONCLUSIÓN FINAL:")
print(f"   Breast Cancer Wisconsin es un excelente dataset para:")
print(f"   - Reforzar conceptos básicos")
print(f"   - Práctica de validación cruzada")
print(f"   - Comparación de algoritmos")
print(f"   - Preparación para datasets más complejos")
print(f"   ")
print(f"   Pero NO debe ser el 'próximo paso principal' después de Wine Quality,")
print(f"   ya que no introduce técnicas significativamente nuevas.")
print(f"   ")
print(f"   🎯 RECOMENDACIÓN: Usar como práctica adicional o bonus session")

---

## 📊 Resumen Comparativo: Breast Cancer vs Progresión del Curso

| Aspecto | Wine Quality | Breast Cancer | Forest Cover | Adult Census |
|---------|--------------|---------------|--------------|--------------|
| **Complejidad** | ⭐ Básica | ⭐⭐ Media | ⭐⭐⭐ Avanzada | ⭐⭐ Media |
| **Features** | 13 (numéricas) | 30 (numéricas) | 54 (mixtos) | 15 (mixtos) |
| **Técnicas Nuevas** | Básicos | Repetidas | Feature Selection | Strings + Missing |
| **Casos de Uso** | Base | Práctica Adicional | Alta Dim. | Texto + Imputación |
| **Tiempo Estimado** | 2-3 horas | 2-3 horas | 4-5 horas | 3-4 horas |

## 🎯 Posición en el Curso

**Breast Cancer Wisconsin** debe usarse como:

### ✅ BUENAS OPCIONES:
- **Práctica adicional** después de dominar Wine Quality
- **Sesión bonus** para reforzar validación cruzada
- **Preparación** para datasets más complejos
- **Comparación de algoritmos** de clasificación

### ❌ NO RECOMENDADO COMO:
- "Próximo paso principal" (no introduce técnicas nuevas)
- Sustituto de Forest Cover Type (faltan feature selection)
- Única práctica antes de Adult Census (necesita más diversidad)

### 🎯 PROGRESIÓN ÓPTIMA:
1. **Wine Quality** → Dominar preprocesamiento básico
2. **Práctica adicional** → Breast Cancer (refuerzo)
3. **Forest Cover Type** → Alta dimensionalidad + feature selection
4. **Adult Census** → Strings + valores faltantes
5. **Heart Disease** → Imputación avanzada

**En resumen**: Breast Cancer es excelente para **práctica adicional** pero no debe reemplazar el salto a **técnicas más avanzadas** como feature selection.